In [ ]:
import MDAnalysis
import numpy as np

pdb="/data/gulab/yzdai/dyz_project1/data/dppc_dopc_chl_280k/snapshot9.99us.gro"
trj="/data/gulab/yzdai/dyz_project1/data/dppc_dopc_chl_280k/dpdochl280k-8us-pbc.xtc"
b = 0
e = 8000
u = MDAnalysis.Universe(pdb, trj)
#define function to cal com of each lipid
#input parameter is sel string for MDAnalysis
#return np array of com position, and no of res
def cal_xycom(sel_lip):
    lip = u.select_atoms(sel_lip)
    lip_com = []
    for i in range(0, len(lip.residues)):
        if i == 0:
            lip_com = lip.residues[i].atoms.center_of_mass()
        else:
            lip_com = np.concatenate((lip_com, lip.residues[i].atoms.center_of_mass()), axis = 0)
    lip_com = lip_com.reshape(len(lip.residues), 3)
    x_zeros = np.zeros(len(lip.residues))
    lip_com[:,2] = x_zeros
   
    lip_resid = list(lip.groupby('resids').keys())
    #print lip_resid
    return lip_com, lip_resid  
#assign each chol to one leaflet
def chol_leaflet(sel_chl):
    #cal the com of the bilayer
    lip_center = u.select_atoms('all').center_of_mass()
    #define list for chol resid storage
    sel_str_upper = []
    sel_str_lower = []
    #sel all chols and calculate com of each chol
    all_chl = u.select_atoms(sel_chl)
    chl_com = []
    for i in range(0, len(all_chl.residues)):
        if i == 0:
            chl_com = all_chl.residues[i].atoms.center_of_mass()
        else:
            chl_com = np.concatenate((chl_com, all_chl.residues[i].atoms.center_of_mass()), axis = 0)
    chl_com = chl_com.reshape(len(all_chl.residues), 3)
    chl_resid = list(all_chl.groupby('resids').keys())
    #assign chol to a leaflet according to the com
    for i in range(0, chl_com.shape[0]):
        diff_pos = chl_com[i, 2] - lip_center[2]
        if diff_pos >= 0:
            sel_str_upper.append(str(chl_resid[i]))
        elif diff_pos < 0:
            sel_str_lower.append(str(chl_resid[i]))
    #check if the total no of chol is correct
    if len(sel_str_upper)+len(sel_str_lower) == 344:
        pass
    else:
        print ("alert, chol no is not 344, .........")

    return sel_str_upper, sel_str_lower

lip_leaflet_list = []
for ts in u.trajectory[b:e]:
    lip_leaflet = [str(0) for i in range(1152)]
    sel_chl_upper, sel_chl_lower = chol_leaflet('resname CHL1')
    lip_resid_upper = list(range(1, 405)) + sel_chl_upper
    lip_resid_lower = list(range(577, 981)) + sel_chl_lower
    lip_resid_upper = [int (x) for x in lip_resid_upper]
    lip_resid_lower = [int (x) for x in lip_resid_lower]
    lip_resid_upper.sort()
    lip_resid_lower.sort()
    # check observations of  each lipid
    box = ts.dimensions[:]
    # record the data to the array, in sequence of resid
    for i in range(0, len(lip_resid_upper)):
        index = int(lip_resid_upper[i])
        lip_leaflet[index-1] = '0'
    for i in range(0, len(lip_resid_lower)):
        index = int(lip_resid_lower[i])
        lip_leaflet[index-1] = '1' # 0 and 1 refer to upper and lower leaflets, respectively

    if len(lip_resid_upper)+len(lip_resid_lower) == 1152:
        pass
    else:
        print ('alert, total res no. are not correct,.................................')
    lip_leaflet_list.append(lip_leaflet)



out = "/data/gulab/yzdai/data4/phase_identification/leaflet/dpdochl280k"
out_leaflet = out+'-leaflet.xvg'
outf_leaflet = open(out_leaflet, 'w')
for i in range(len(lip_leaflet_list)):    
    print(str(i), file = outf_leaflet, end=' ')
    for j in range(0, 1152):
        print('%s' % (lip_leaflet_list[i][j]), file = outf_leaflet, end=' ')
    print('\n', file=outf_leaflet, end='')